# 01 — Ingestion & Bronze Layer

Download OpenF1 data, save **raw Bronze JSONL**, and generate ingestion evidence (manifest, row counts, schema reports).

| Endpoint | Role |
|----------|------|
| `session_result` | **Required** for Gold target `points_finish` |
| `starting_grid` | **Optional** (may be 404 / empty) |
| Other session endpoints | Per-race telemetry and results |
| `meetings`, `sessions` | Global calendar discovery |

**Run order:** `SMOKE_TEST=True` first → then full run with `SMOKE_TEST=False` and `USE_GOOGLE_DRIVE=True`.

> Full ingestion can take **hours**. Use Drive persistence. Official MBA evidence must come from **Colab Pro Plus**.


## Colab setup (required every session)

This cell clones/updates the repo, installs `requirements.txt` and `pip install -e .`, mounts Drive, and sets `OPENF1_DATA_ROOT`. Do not import `openf1_pipeline` before this cell completes.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Updating repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Start Spark (PySpark — local session, no Databricks)


In [2]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


INFO:openf1_pipeline.utils.spark:Spark session started: openf1-big-data-pipeline


Spark version: 4.0.2


## Bronze configuration & path check


In [3]:
from openf1_pipeline.config import (
    ENDPOINTS,
    GLOBAL_ENDPOINTS,
    SESSION_ENDPOINTS,
    SEASONS,
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_project_root,
    get_schemas_dir,
)
from openf1_pipeline.ingestion.ingest import run_bronze_ingestion, summarize_manifest
from openf1_pipeline.bronze.build_bronze import generate_bronze_reports

print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("ENDPOINTS:", ENDPOINTS)
print("GLOBAL_ENDPOINTS:", GLOBAL_ENDPOINTS)
print("SESSION_ENDPOINTS:", SESSION_ENDPOINTS)
print("BRONZE_DIR:", get_bronze_dir())
print("MANIFESTS_DIR:", get_manifests_dir())
print("DATA_QUALITY_REPORTS_DIR:", get_data_quality_reports_dir())
print("SCHEMAS_DIR:", get_schemas_dir())
print("session_result: REQUIRED for Gold points_finish")
print("starting_grid: OPTIONAL — failures are logged and ingestion continues")


PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
ENDPOINTS: ['meetings', 'sessions', 'drivers', 'laps', 'pit', 'weather', 'position', 'race_control', 'session_result', 'starting_grid']
GLOBAL_ENDPOINTS: ['meetings', 'sessions']
SESSION_ENDPOINTS: ['drivers', 'laps', 'pit', 'weather', 'position', 'race_control', 'session_result', 'starting_grid']
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
MANIFESTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests
DATA_QUALITY_REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
SCHEMAS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/schemas
session_result: REQUIRED for Gold points_finish
starting_grid: OPTIONAL — failures are logged and ingestion continues


In [4]:
# Step 1: smoke test. Step 2: set SMOKE_TEST = False for full 2023–2025 evidence.
SMOKE_TEST = False
MAX_SESSIONS = 2 if SMOKE_TEST else None
INGEST_SEASONS = [2024] if SMOKE_TEST else SEASONS

# WARNING: CLEAR_BRONZE_OUTPUTS=True deletes all Bronze JSONL and re-ingests from API (slow).
CLEAR_BRONZE_OUTPUTS = False
BRONZE_REPORT_ENGINE = "spark"
ALLOW_FALLBACK = False

print(f"SMOKE_TEST={SMOKE_TEST}, seasons={INGEST_SEASONS}, max_sessions={MAX_SESSIONS}")
print(f"CLEAR_BRONZE_OUTPUTS={CLEAR_BRONZE_OUTPUTS}, ALLOW_FALLBACK={ALLOW_FALLBACK}")
if not SMOKE_TEST:
    print("WARNING: Full ingestion may take a long time. Ensure USE_GOOGLE_DRIVE=True above.")


SMOKE_TEST=False, seasons=[2023, 2024, 2025], max_sessions=None
CLEAR_BRONZE_OUTPUTS=False, ALLOW_FALLBACK=False


## Optional: clean Bronze outputs


In [ ]:
from openf1_pipeline.utils.cleanup import clean_bronze_layer_outputs

if CLEAR_BRONZE_OUTPUTS:
    print("WARNING: Deleting Bronze data and Bronze reports — re-ingestion required.")
    clean_bronze_layer_outputs()
else:
    print("Skipping Bronze cleanup (CLEAR_BRONZE_OUTPUTS=False).")


## Run Bronze ingestion


In [ ]:
manifest_df = run_bronze_ingestion(
    seasons=INGEST_SEASONS,
    endpoints=ENDPOINTS,
    bronze_dir=get_bronze_dir(),
    manifests_dir=get_manifests_dir(),
    max_sessions=MAX_SESSIONS,
)

manifest_summary = summarize_manifest(manifest_df)
manifest_summary


## Endpoint status summary


In [ ]:
print("=== Manifest status counts ===")
print(manifest_summary["status_counts"])
print("\n=== Success endpoints ===")
print(manifest_summary["success_endpoints"])
print("\n=== Failed endpoints (continued) ===")
print(manifest_summary["failed_endpoints"])
print("\n=== Row counts by endpoint ===")
for ep, n in sorted(manifest_summary["row_counts_by_endpoint"].items()):
    print(f"  {ep}: {n}")
print(f"\nsession_result rows: {manifest_summary['session_result_total_rows']}")
print(f"starting_grid rows: {manifest_summary['starting_grid_total_rows']}")
if manifest_summary["session_result_total_rows"] == 0 and not SMOKE_TEST:
    print("WARNING: session_result has zero rows — check manifest before Silver/Gold.")
manifest_df.groupby(["endpoint", "status"]).size().unstack(fill_value=0)


## Generate Bronze evidence reports (Spark-first)


In [ ]:
report_result = generate_bronze_reports(
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    schemas_dir=get_schemas_dir(),
    engine=BRONZE_REPORT_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)
report_result


## Manifest ↔ Bronze files reconciliation

Cross-checks the **effective** Bronze manifest (original ingestion overlaid with any
successful retry rows) against the JSONL inventory on disk. Produces:

- `reports/data_quality/bronze_manifest_file_reconciliation.csv`
- `reports/data_quality/bronze_manifest_file_reconciliation_summary.csv`

This is the canonical place to detect stale files (e.g. smoke leftovers), row-count drift,
manifest-success-but-file-missing, and failed-but-file-present states **before** Silver runs.

The cell below auto-detects whether a prior `ingestion_retry_manifest.csv` exists. If retry
recovered any rows it reconciles against `artifacts/manifests/ingestion_manifest_effective.csv`
(merging original + retry without touching the original manifest); otherwise it reconciles
against the original `ingestion_manifest.csv`.



In [16]:
from openf1_pipeline.bronze.build_bronze import (
    generate_bronze_reconciliation_reports,
    generate_bronze_reports
)
from openf1_pipeline.ingestion.ingest import (
    summarize_retry_manifest,
    write_effective_manifest_after_retry,
)
import pandas as pd

# Ensure report_result is defined for this cell's execution.
# In a standard notebook flow, this would be defined in cell FMIGOm2t2vxK.
report_result = generate_bronze_reports(
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    schemas_dir=get_schemas_dir(),
    engine=BRONZE_REPORT_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)

_original_manifest_path = get_manifests_dir() / "ingestion_manifest.csv"
_retry_manifest_path = get_manifests_dir() / "ingestion_retry_manifest.csv"

# Defensive: if a retry manifest exists with newly_successful > 0, reconciliation
# MUST use the effective merged manifest. Otherwise retry-recovered files would
# appear as failed_but_file_present against the original manifest.
if _retry_manifest_path.is_file():
    _retry_preview = pd.read_csv(_retry_manifest_path)
    _retry_preview_summary = summarize_retry_manifest(_retry_preview)
    if _retry_preview_summary["newly_successful"] > 0:
        _effective = write_effective_manifest_after_retry(
            manifest_path=_original_manifest_path,
            retry_manifest_path=_retry_manifest_path,
            manifests_dir=get_manifests_dir(),
        )
        _manifest_path_for_reconciliation = _effective["path"]
        print(
            "[reconciliation] Using effective merged manifest "
            f"({_retry_preview_summary['newly_successful']} retry successes overlay original):"
        )
        print(f"  {_manifest_path_for_reconciliation}")
        print(
            "  rows_from_original=" + str(_effective["counts"]["rows_from_original"]) +
            ", rows_from_retry=" + str(_effective["counts"]["rows_from_retry"]) +
            ", retry_keys_replaced=" + str(_effective["counts"]["retry_keys_replaced"])
        )
    else:
        _manifest_path_for_reconciliation = _original_manifest_path
        print(
            "[reconciliation] Retry manifest present but no newly successful rows."
        )
        print(f"  Using original manifest: {_manifest_path_for_reconciliation}")
else:
    _manifest_path_for_reconciliation = _original_manifest_path
    print(f"[reconciliation] Using original manifest: {_manifest_path_for_reconciliation}")

reconciliation = generate_bronze_reconciliation_reports(
    manifest_path=_manifest_path_for_reconciliation,
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    row_counts_df=pd.read_csv(report_result["paths"]["bronze_row_counts"]),
)
reconciliation_df = reconciliation["df"]
reconciliation_totals = reconciliation["summary"]

print("Reconciliation totals:")
for k, v in reconciliation_totals.items():
    print(f"  {k}: {v}")

display(
    reconciliation_df.groupby("reconciliation_status").size().reset_index(name="count")
)

non_matched = reconciliation_df[
    reconciliation_df["reconciliation_status"] != "matched"
]
if not non_matched.empty:
    print(f"\nNon-matched rows: {len(non_matched)}")
    display(non_matched.head(40))

stale = reconciliation_df[
    reconciliation_df["reconciliation_status"] == "stale_file_not_in_success_manifest"
]
failed_with_file = reconciliation_df[
    reconciliation_df["reconciliation_status"] == "failed_manifest_file_exists"
]
row_mismatches = reconciliation_df[
    reconciliation_df["reconciliation_status"] == "row_count_mismatch"
]
missing_files = reconciliation_df[
    reconciliation_df["reconciliation_status"] == "manifest_success_missing_file"
]

if len(stale):
    print(f"\nStale files (no success manifest row): {len(stale)}")
    display(stale[["endpoint", "year", "session_key", "file_path", "file_row_count"]].head(40))
if len(failed_with_file):
    print(f"\nFailed-but-file-present rows: {len(failed_with_file)}")
    display(failed_with_file[[
        "endpoint", "year", "session_key", "manifest_status", "file_row_count", "notes"
    ]].head(40))
if len(row_mismatches):
    print(f"\nRow-count mismatches: {len(row_mismatches)}")
    display(row_mismatches[[
        "endpoint", "year", "session_key",
        "manifest_record_count", "file_row_count", "row_count_delta",
    ]].head(40))
if len(missing_files):
    print(f"\nManifest-success but missing on disk: {len(missing_files)}")
    display(missing_files[[
        "endpoint", "year", "session_key", "manifest_output_path", "manifest_record_count",
    ]].head(40))

blocking = (
    reconciliation_totals.get("stale_files", 0)
    + reconciliation_totals.get("failed_but_file_present", 0)
    + reconciliation_totals.get("row_mismatches", 0)
    + reconciliation_totals.get("missing_files", 0)
)
if blocking:
    print(
        "\nWARNING: Bronze reconciliation found inconsistencies. "
        "If stale files exist, do not proceed to Silver until choosing a cleanup or retry policy. "
        "Use the targeted retry section below for 429-related failures, or surgically delete "
        "stale Bronze JSONL files before Silver."
    )
else:
    print("\nOK: Bronze reconciliation is clean — safe to proceed to Silver.")

INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_file_inventory.csv (609 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_row_counts.csv (609 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_schema_report.csv (106 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/schemas/bronze_schema_report.csv (106 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_schema_drift.csv (106 rows)
INFO:openf1_pipeline.bronze.build_bronze:Bronze Spark reports generated: {'engine': 'spark', 'bronze_files': 609, 'total_rows': 148184, 'endpoints': ['drivers', 'laps', 'meetings', 'pit', 'position', 'race_control', 'session_result', 'sessions', 'weather'], 'schema_columns': 106, 

[reconciliation] Using effective merged manifest (134 retry successes overlay original):
  /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/ingestion_manifest_effective.csv
  rows_from_original=564, rows_from_retry=154, retry_keys_replaced=154
Reconciliation totals:
  matched: 609
  stale_files: 0
  missing_files: 0
  row_mismatches: 0
  failed_but_file_present: 0
  optional_missing: 89
  manifest_failed_no_file: 20
  unknown: 0
  total_rows_reconciled: 718


,reconciliation_status,count
0,manifest_failed_no_file,20
1,matched,609
2,optional_missing,89



Non-matched rows: 109


,endpoint,year,session_key,manifest_status,manifest_record_count,manifest_output_path,file_exists,file_path,file_row_count,row_count_delta,reconciliation_status,issue_type,notes
0,drivers,2023,9086.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
1,laps,2023,9086.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
2,pit,2023,7779.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
3,pit,2023,7787.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
4,pit,2023,7953.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
5,pit,2023,9069.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
6,pit,2023,9070.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
7,pit,2023,9078.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
8,pit,2023,9086.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...
9,pit,2023,9094.0,failed,0,/content/drive/MyDrive/openf1_big_data_pipelin...,False,None,NaN,NaN,manifest_failed_no_file,manifest_only,required session endpoint failed; consider tar...



OK: Bronze reconciliation is clean — safe to proceed to Silver.


## DuckDB validation (Bronze CSV evidence)


In [ ]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_bronze_with_duckdb,
)

bronze_duckdb = validate_bronze_with_duckdb(get_data_quality_reports_dir())
duckdb_bronze_paths = save_duckdb_validation_reports(
    bronze_duckdb, get_data_quality_reports_dir(), prefix="bronze"
)
duckdb_bronze_paths


## Inspect outputs


In [ ]:
import pandas as pd

row_counts = pd.read_csv(report_result["paths"]["bronze_row_counts"])
schema_report = pd.read_csv(report_result["paths"]["bronze_schema_report"])
schema_drift = pd.read_csv(report_result["paths"]["bronze_schema_drift"])

print("Row counts by endpoint:")
display(row_counts.groupby("endpoint")["row_count"].sum().reset_index())
display(schema_report.head(10))
drift_flags = schema_drift[schema_drift["possible_schema_drift_flag"] == True]
print(f"Schema drift flags: {len(drift_flags)}")
if len(drift_flags):
    display(drift_flags.head(15))


## Optional: targeted retry for failed session endpoints

Use this section only when the main ingestion above left failures (typically HTTP 429 rate-limit
storms) and you want to recover the affected sessions without re-running full Bronze.

- Reads `artifacts/manifests/ingestion_manifest.csv` and retries failed **session-level**
  endpoint calls with a slower base sleep (default `RETRY_SLEEP_SECONDS = 3.0`).
- Writes only the target endpoint/session JSONL files. Successful Bronze data is **not** touched.
- `starting_grid` is **excluded by default** (optional endpoint, consistently 404). Set
  `RETRY_INCLUDE_OPTIONAL = True` to include it.
- Original `ingestion_manifest.csv` is **not** modified. Retry results land in
  `artifacts/manifests/ingestion_retry_manifest.csv`.
- After retry, Bronze Spark reports and DuckDB validation are regenerated so
  `bronze_row_counts.csv`, `bronze_schema_report.csv`, `bronze_schema_drift.csv`, and the
  `duckdb_bronze_*` CSVs reflect the new file inventory.

The retry **does not run** unless `RUN_TARGETED_RETRY = True`. Leave it `False` to skip.


In [17]:
# Targeted retry configuration — does nothing unless RUN_TARGETED_RETRY=True.
RUN_TARGETED_RETRY = True
RETRY_SLEEP_SECONDS = 3.0          # base inter-request sleep (seconds)
RETRY_MAX_PER_REQUEST = 5          # OpenF1Client per-request retry budget
RETRY_INCLUDE_OPTIONAL = False     # set True to also retry starting_grid
# Restrict the retry to specific endpoints (None = all required session endpoints).
RETRY_ENDPOINTS = None             # e.g. ["session_result", "pit"]

# Optional: delete known stale smoke-run JSONL files from Drive before/after retry.
# Paths are relative to BRONZE_DIR. Set DELETE_STALE_SMOKE_FILES=True only when you
# have confirmed the retry replaced them OR you want a manifest-clean Drive state.
DELETE_STALE_SMOKE_FILES = False
STALE_SMOKE_BRONZE_FILES = [
    "session_result/year=2024/session_key=9472.jsonl",
    "session_result/year=2024/session_key=9480.jsonl",
    "pit/year=2024/session_key=9480.jsonl",
]

print(f"RUN_TARGETED_RETRY={RUN_TARGETED_RETRY}")
print(f"RETRY_SLEEP_SECONDS={RETRY_SLEEP_SECONDS}, RETRY_MAX_PER_REQUEST={RETRY_MAX_PER_REQUEST}")
print(f"RETRY_INCLUDE_OPTIONAL={RETRY_INCLUDE_OPTIONAL}, RETRY_ENDPOINTS={RETRY_ENDPOINTS}")
print(f"DELETE_STALE_SMOKE_FILES={DELETE_STALE_SMOKE_FILES}")


RUN_TARGETED_RETRY=True
RETRY_SLEEP_SECONDS=3.0, RETRY_MAX_PER_REQUEST=5
RETRY_INCLUDE_OPTIONAL=False, RETRY_ENDPOINTS=None
DELETE_STALE_SMOKE_FILES=False


### Run the targeted retry

This cell is a no-op unless `RUN_TARGETED_RETRY = True`. The retry writes
`artifacts/manifests/ingestion_retry_manifest.csv` next to the original manifest.


In [7]:
from openf1_pipeline.ingestion.ingest import (
    delete_stale_bronze_files,
    retry_failed_session_endpoints,
    summarize_retry_manifest,
)

retry_df = None
retry_summary = None

if RUN_TARGETED_RETRY:
    # Optional: clean known stale smoke-run Drive files before retry.
    if DELETE_STALE_SMOKE_FILES:
        deleted = delete_stale_bronze_files(
            bronze_dir=get_bronze_dir(),
            paths=STALE_SMOKE_BRONZE_FILES,
        )
        print(f"Deleted {len(deleted)} stale Bronze file(s):")
        for p in deleted:
            print("  -", p)

    retry_df = retry_failed_session_endpoints(
        manifest_path=get_manifests_dir() / "ingestion_manifest.csv",
        bronze_dir=get_bronze_dir(),
        manifests_dir=get_manifests_dir(),
        endpoints_to_retry=RETRY_ENDPOINTS,
        include_optional=RETRY_INCLUDE_OPTIONAL,
        sleep_seconds=RETRY_SLEEP_SECONDS,
        max_retries_per_request=RETRY_MAX_PER_REQUEST,
    )

    retry_summary = summarize_retry_manifest(retry_df)
    print("Retry summary:")
    print(retry_summary)
    if not retry_df.empty:
        display(retry_df.groupby(["endpoint", "status"]).size().unstack(fill_value=0))
else:
    print("Targeted retry skipped (RUN_TARGETED_RETRY=False).")


NameError: name 'RUN_TARGETED_RETRY' is not defined

### Regenerate Bronze reports, reconciliation, and DuckDB validation after retry

Only runs when `RUN_TARGETED_RETRY = True` and the retry actually attempted rows.

This refresh:

1. Regenerates Bronze Spark reports (`bronze_file_inventory.csv`, `bronze_row_counts.csv`,
   `bronze_schema_report.csv`, `bronze_schema_drift.csv`).
2. Builds the **effective post-retry manifest**
   `artifacts/manifests/ingestion_manifest_effective.csv` by overlaying retry successes on
   the original ingestion manifest. The original `ingestion_manifest.csv` is preserved
   unchanged for auditability.
3. Runs reconciliation **against the effective merged manifest** so retry-recovered files
   are correctly classified as `matched` (instead of `failed_but_file_present`).
4. Refreshes all `duckdb_bronze_*` CSVs including the reconciliation cross-checks.


In [18]:
from openf1_pipeline.ingestion.ingest import (
    write_effective_manifest_after_retry,
)

if RUN_TARGETED_RETRY and retry_df is not None and not retry_df.empty:
    refreshed_report = generate_bronze_reports(
        bronze_dir=get_bronze_dir(),
        data_quality_reports_dir=get_data_quality_reports_dir(),
        schemas_dir=get_schemas_dir(),
        engine=BRONZE_REPORT_ENGINE,
        spark=spark,
        allow_fallback=ALLOW_FALLBACK,
    )

    # Build a durable effective manifest from original + retry (original CSV is preserved).
    effective_manifest = write_effective_manifest_after_retry(
        manifest_path=get_manifests_dir() / "ingestion_manifest.csv",
        retry_manifest_path=get_manifests_dir() / "ingestion_retry_manifest.csv",
        manifests_dir=get_manifests_dir(),
    )
    effective_manifest_path = effective_manifest["path"]
    print("Effective manifest written:")
    print(f"  path: {effective_manifest_path}")
    for k, v in effective_manifest["counts"].items():
        print(f"  {k}: {v}")

    print(
        "\nReconciliation manifest used: "
        + str(effective_manifest_path)
        + " (effective merged manifest)"
    )

    refreshed_reconciliation = generate_bronze_reconciliation_reports(
        manifest_path=effective_manifest_path,
        bronze_dir=get_bronze_dir(),
        data_quality_reports_dir=get_data_quality_reports_dir(),
        row_counts_df=pd.read_csv(refreshed_report["paths"]["bronze_row_counts"]),
    )

    refreshed_bronze_duckdb = validate_bronze_with_duckdb(get_data_quality_reports_dir())
    refreshed_duckdb_paths = save_duckdb_validation_reports(
        refreshed_bronze_duckdb, get_data_quality_reports_dir(), prefix="bronze"
    )

    print("\nRefreshed Bronze report summary:")
    print(refreshed_report["summary"])
    print("\nRefreshed reconciliation totals:")
    for k, v in refreshed_reconciliation["summary"].items():
        print(f"  {k}: {v}")
    print("\nRefreshed DuckDB Bronze paths:")
    for k, v in refreshed_duckdb_paths.items():
        print(f"  {k}: {v}")

    blocking_after_retry = (
        refreshed_reconciliation["summary"].get("stale_files", 0)
        + refreshed_reconciliation["summary"].get("failed_but_file_present", 0)
        + refreshed_reconciliation["summary"].get("row_mismatches", 0)
        + refreshed_reconciliation["summary"].get("missing_files", 0)
    )
    if blocking_after_retry == 0:
        print("\nOK: post-retry reconciliation is clean — safe to proceed to Silver.")
    else:
        print(
            "\nWARNING: post-retry reconciliation still shows inconsistencies. "
            "Review bronze_manifest_file_reconciliation.csv before Silver."
        )
else:
    print("Skipping Bronze report refresh — no retry rows or retry disabled.")


Skipping Bronze report refresh — no retry rows or retry disabled.


### Verify session_result coverage after retry

Compares the original manifest, the retry manifest, the effective merged manifest, and the
current Drive file inventory. The effective manifest is the authoritative post-retry view.


In [19]:
if RUN_TARGETED_RETRY and retry_df is not None and not retry_df.empty:
    original_manifest = pd.read_csv(get_manifests_dir() / "ingestion_manifest.csv")
    effective_manifest_df = pd.read_csv(
        get_manifests_dir() / "ingestion_manifest_effective.csv"
    )
    refreshed_inventory = pd.read_csv(
        get_data_quality_reports_dir() / "bronze_file_inventory.csv"
    )

    sr_orig_ok = (
        (original_manifest["endpoint"] == "session_result")
        & (original_manifest["status"] == "success")
    ).sum()
    sr_retry_ok = (
        (retry_df["endpoint"] == "session_result")
        & (retry_df["status"] == "success")
    ).sum()
    sr_effective_ok = (
        (effective_manifest_df["endpoint"] == "session_result")
        & (effective_manifest_df["status"] == "success")
    ).sum()
    sr_files = (refreshed_inventory["endpoint"] == "session_result").sum()

    print(f"session_result manifest success (original run): {sr_orig_ok}")
    print(f"session_result manifest success (retry only): {sr_retry_ok}")
    print(f"session_result manifest success (effective merged): {sr_effective_ok}")
    print(f"session_result JSONL files on Drive (after retry): {sr_files}")
    print(f"=> effective session_result coverage: {sr_effective_ok} sessions")
    if sr_effective_ok != sr_orig_ok + sr_retry_ok:
        print(
            "  NOTE: effective != original + retry — expected if retry attempts overlapped "
            "with already-successful original rows."
        )
else:
    print("Skipping coverage verification (retry not run or retry was empty).")


Skipping coverage verification (retry not run or retry was empty).
